# What imagery exists over this fire?

**Finding satellite scenes with `georest.cloud.stac`**

After a wildfire, the first question about satellite imagery is practical: *which pictures of this
place were taken, when, and how cloudy were they?* Answering it used to mean visiting a
different website for each satellite. Today the imagery sits in cloud storage, and catalogs
list it using a shared standard called **STAC** (SpatioTemporal Asset Catalog). A STAC catalog
holds **collections** (one per kind of data, such as "Sentinel-2 surface reflectance"). A
collection holds **items**, which this notebook calls **scenes**: one image of one place on one
day. A scene has **assets**, the files that make it up (one per color band, plus a preview).

`georest.cloud.stac` searches those catalogs and gives back small tables of scenes and links.
**It never reads image pixels.** Reading pixels is the job of libraries built for it, and this
notebook shows the hand-off in the last step.

You will:

1. get the perimeter of the 2020 Cameron Peak fire in Colorado from the Forest Service's
   Enterprise Data Warehouse (EDW), which publishes its data as ArcGIS services,
2. see which STAC catalogs georest knows and pick a collection,
3. search for scenes over the perimeter in the three months after ignition,
4. chart when they were taken and how cloudy they were, and map where they fall,
5. list one scene's files, and
6. open one file with `rasterio` and look at the fire.

```bash
pip install pandas georest plotly folium rasterio
```

In [2]:
import folium
import pandas as pd
import plotly.express as px
import plotly.io as pio
import rasterio
from rasterio.warp import transform_bounds
from rasterio.windows import Window, from_bounds

from georest.cloud import stac
from georest.esri import edw

pd.set_option("display.max_colwidth", None)   # show text in full, never cut with "..."
pio.renderers.default = "notebook_connected"   # interactive here and on georest.io

ModuleNotFoundError: No module named 'pandas'

## The fire

The Monitoring Trends in Burn Severity (MTBS) program maps the perimeter of every large fire in
the United States, and the EDW service `EDW_MTBS_01` publishes it. That service has a layer for
each year plus one called "Burned Area Boundaries (All Years)". Layer numbers shift when EDW
republishes, so the cell below finds the layer by its name.

A fire perimeter drawn at survey detail has tens of thousands of points, more than a catalog
search accepts (10,000). `max_allowable_offset=0.001` asks EDW to simplify the shape before it
sends it, to within about 100 meters. That is plenty for deciding which scenes cover the fire.

In [ ]:
layers = edw.get_service_info("EDW_MTBS_01")["layers"]
layer = next(l["id"] for l in layers if l["name"] == "Burned Area Boundaries (All Years)")

fire = edw.query_features("EDW_MTBS_01", layer,
                          where="fire_name = 'CAMERON PEAK' AND year = 2020",
                          out_fields="fire_name,year,acres,ig_date",
                          max_allowable_offset=0.001)
props = fire["features"][0]["properties"]
ignition = pd.to_datetime(str(props["ig_date"]), format="%Y%m%d")   # EDW stores 20200813
print(f"{props['fire_name'].title()}, {props['year']}: {props['acres']:,} acres, "
      f"ignited {ignition:%B %d, %Y}")

## Where to look

`stac.catalogs()` lists the catalogs georest knows. The `assets` column says how a scene's files
are read: `open` links work in any browser or library, while `sign` and `earthdata` links need
a token first. This notebook stays on `earth-search`, an open catalog of imagery held in
Amazon's cloud, so everything below runs without an account.

In [ ]:
stac.catalogs()[["name", "title", "assets"]]

Within a catalog, `stac.collections` lists collections and can filter them by a word. Sentinel-2
is a pair of European satellites that photograph the whole land surface every few days at
10-meter detail; "L2A" means the images are already corrected for the atmosphere, which is
the version most people want.

In [ ]:
stac.collections("earth-search", "sentinel")[["id", "title", "start"]]

## Search

`stac.search` takes the catalog, the collection, a place and a date range. The place here is the
fire's GeoJSON exactly as EDW returned it, passed to `intersects`, so a scene is returned if
its footprint (the outline of the ground it covers) overlaps the perimeter. (When you only have a
rectangle, `bbox=(west, south, east, north)` does the same job.) `max_cloud_cover=30` leaves out
scenes more than 30% hidden by cloud.

Two fields of the result matter whenever a search might be large. `attrs["matched"]` is how many
scenes the server says fit, and `attrs["truncated"]` is `True` if `max_items` stopped the fetch
before all of them were returned.

In [ ]:
window = f"{ignition:%Y-%m-%d}/{ignition + pd.Timedelta(days=90):%Y-%m-%d}"
scenes = stac.search("earth-search", "sentinel-2-l2a", intersects=fire,
                     datetime=window, max_cloud_cover=30, max_items=200)
print(f"{len(scenes)} scenes returned; the server matched {scenes.attrs['matched']}; "
      f"truncated: {scenes.attrs['truncated']}")
scenes[["id", "datetime", "cloud_cover", "platform"]].head()

## When could you see it?

Each dot is one scene. Sentinel-2 passes over a given spot every few days, so the dots come in
runs, and the gaps are stretches when cloud or smoke kept every picture out. One caveat: a scene
covers a large square, and `cloud_cover` is a figure for the whole square. The fire itself can
be clearer or cloudier than that number says.

In [ ]:
chart = scenes.assign(date=pd.to_datetime(scenes["datetime"]))
fig = px.scatter(chart, x="date", y="cloud_cover", color="platform", hover_name="id",
                 labels={"date": "", "cloud_cover": "Cloud cover (%)", "platform": "Satellite"},
                 title="Sentinel-2 scenes over the Cameron Peak fire, after ignition")
fig.update_traces(marker_size=9)
fig.show()

## Where are the scenes?

`stac.footprints` turns the search result into GeoJSON outlines, one per scene, which a map
can draw. Many scenes share an outline because the satellites revisit the same tiles (the fixed
squares Sentinel-2 divides the land into), so the map shows a few overlapping squares. Hover
over the perimeter for the fire's name and size.

In [ ]:
outlines = stac.footprints(scenes)

m = folium.Map(tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}",
               attr="Tiles &copy; Esri")
folium.GeoJson(outlines, name="Scene footprints",
               style_function=lambda f: {"color": "#2563eb", "weight": 1, "fillOpacity": 0.03}).add_to(m)
folium.GeoJson(fire, name="Cameron Peak perimeter",
               tooltip=folium.GeoJsonTooltip(fields=["fire_name", "acres"]),
               style_function=lambda f: {"color": "#c2410c", "weight": 2, "fillOpacity": 0.25}).add_to(m)
m.fit_bounds(m.get_bounds())
m

## One scene's files

A search row says *that* a scene exists. `stac.assets` lists the files that make it up, one row
per file: its `key` (the name within the scene), its type, and `href`, the link. Most are
**COGs** (Cloud Optimized GeoTIFFs), image files laid out so a program can fetch just the part
it needs over the internet without downloading the rest. The `visual` file is a ready-made
true-color picture; the others are individual color bands for analysis.

Here is the clearest scene in the search.

In [ ]:
best = scenes.sort_values("cloud_cover").iloc[0]
print(f"{best['id']}, {pd.to_datetime(best['datetime']):%B %d}, {best['cloud_cover']:.1f}% cloud")
files = stac.assets(best["catalog"], best["collection"], best["id"])
files[files["type"].str.contains("geotiff", na=False)][["key", "title", "href"]].head(8)

## Reading the pixels (not georest's job)

Everything so far was *finding* imagery. georest stops at the link, and this last cell is where
`rasterio`, a general-purpose library for geospatial image files, takes over. It opens the
`visual` COG straight from the cloud, asks for only the rectangle around the fire, and reads it
at lower resolution so the download is small. The scene's image uses a metric map projection,
so the perimeter's longitude and latitude box is converted first.

One scene is one tile, and the Cameron Peak fire is large enough to cross a tile edge, so the
picture shows the part of the fire that falls inside this scene. Joining tiles into one image is a
job for `rasterio` and its relatives, not for this guide.

In [ ]:
def positions(coords):
    """Every (lon, lat) point in a GeoJSON polygon's nested coordinate lists."""
    if isinstance(coords[0], (int, float)):
        yield coords
    else:
        for part in coords:
            yield from positions(part)


points = list(positions(fire["features"][0]["geometry"]["coordinates"]))
lons, lats = [p[0] for p in points], [p[1] for p in points]
href = files.loc[files["key"] == "visual", "href"].iloc[0]

with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif"):
    with rasterio.open(href) as src:
        left, bottom, right, top = transform_bounds("EPSG:4326", src.crs,
                                                    min(lons), min(lats), max(lons), max(lats))
        box = from_bounds(left, bottom, right, top, src.transform).round_offsets().round_lengths()
        box = box.intersection(Window(0, 0, src.width, src.height))   # the scene may cover only part
        width = 500
        height = max(1, round(box.height * width / box.width))
        picture = src.read(window=box, out_shape=(3, height, width))
print(f"read a {int(box.width)} x {int(box.height)} pixel window, shown at {width} x {height}")

fig = px.imshow(picture.transpose(1, 2, 0), title="True-color view around the fire (Sentinel-2 visual)")
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
fig.update_traces(hoverinfo="skip", hovertemplate=None)
fig.show()

## Where to go next

- **Another catalog.** Swap `"earth-search"` for `"planetary-computer"`, whose files need a
  short-lived read token: `stac.sign(href, collection)` adds it. The tokens expire within
  hours, which is why this notebook does not use that catalog.
- **Another collection.** `stac.collections("earth-search")` lists them all, and
  `stac.search` works the same way with a different id, for example Landsat or the
  National Agriculture Imagery Program (NAIP), aerial photos at about one-meter detail.
- **Any STAC API.** Pass its root URL where the catalog name goes.
- **An AI assistant.** The georest MCP server (MCP, the Model Context Protocol, is a standard
  for giving a language model tools) offers `stac_collections`, `stac_search` and
  `stac_assets`, which return the same links and metadata.